# 03 — Compressed text

The third shape, and the one that catches everyone once. The file arrives
**gzip-compressed** — smaller to send, which is why almost every data service does it.

The data inside is ordinary text. The compression is on the outside.

## 1. What we're fetching

```bash
https://www.ndbc.noaa.gov/data/historical/stdmet/46092h2019.txt.gz
```

Hourly wind and wave observations from an actual buoy 10 km off Monterey Bay. The
`.gz` at the end is gzip.

In [1]:
import requests

NDBC_URL = "https://www.ndbc.noaa.gov/data/historical/stdmet/46092h2019.txt.gz"

r = requests.get(NDBC_URL, timeout=30)
r.raise_for_status()
print(r.status_code, len(r.content), "bytes")
print("content type:", r.headers.get("Content-Type"))

200 76016 bytes
content type: application/x-gzip


## 2. The mistake everybody makes

`r.text` looks like it should work. It does not.

`r.text` takes the **bytes** and *decodes them as text*. Gzip bytes are not text —
they are a compressed stream, and decoding them gives you a few hundred characters of
mojibake followed by nothing. It raises nothing. You would have to read the output to
notice.

`r.content` is the bytes untouched. You un-compress those yourself.

In [2]:
import gzip

wrong = r.text[:60]
print("r.text gives you:")
print("   ", repr(wrong))
print()
print("  That is the gzip header, decoded as if it were language. Silently useless.")
print()

text = gzip.decompress(r.content).decode()   # the actual file
lines = text.splitlines()
print("r.content + gzip gives you:", len(lines), "lines")
for ln in lines[:3]:
    print("   ", ln[:74])

r.text gives you:
    '\x1f�\x08\x08s�B^\x00\x0346092h2019.txt\x00��M�uA�����+\x0e������94\\\r�\x07\r4�\r544,��'

  That is the gzip header, decoded as if it were language. Silently useless.

r.content + gzip gives you: 7837 lines
    #YY  MM DD hh mm WDIR WSPD GST  WVHT   DPD   APD MWD   PRES  ATMP  WTMP  D
    #yr  mo dy hr mn degT m/s  m/s     m   sec   sec degT   hPa  degC  degC  d
    2019 01 05 19 12 130 10.3 99.0 99.00 99.00 99.00 999 1005.7  13.9  13.1 99


## 3. The second thing about this file

Line 0 is the column **names**. Line 1 is the **units**. Data starts at line 2.

Same shape as notebook 01's trap, and the mistake is identical: parse the units line
instead of the names line and your columns come out called `#yr`, `mo`, `dy`, `hr`,
`mn`. You will not find out for an hour, when something you thought was a column
turns out not to exist.

There is no pandas option that reads fixed-width NDBC text for you. You read the two
header lines and split the rest.

In [3]:
cols = [c.lstrip("#") for c in lines[0].split()]      # the NAMES line
rows = [ln.split() for ln in lines[2:] if ln.strip()]  # from line 2 onwards

print("columns:", cols)
print("rows   :", len(rows))
print("first  :", rows[0])

columns: ['YY', 'MM', 'DD', 'hh', 'mm', 'WDIR', 'WSPD', 'GST', 'WVHT', 'DPD', 'APD', 'MWD', 'PRES', 'ATMP', 'WTMP', 'DEWP', 'VIS', 'TIDE']
rows   : 7835
first  : ['2019', '01', '05', '19', '12', '130', '10.3', '99.0', '99.00', '99.00', '99.00', '999', '1005.7', '13.9', '13.1', '999.0', '99.0', '99.00']


In [4]:
import pandas as pd

df = pd.DataFrame(rows, columns=cols)
print(df.head(3).to_string(index=False))

  YY MM DD hh mm WDIR WSPD  GST  WVHT   DPD   APD MWD   PRES ATMP WTMP  DEWP  VIS  TIDE
2019 01 05 19 12  130 10.3 99.0 99.00 99.00 99.00 999 1005.7 13.9 13.1 999.0 99.0 99.00
2019 01 05 20 12  160 11.8 99.0 99.00 99.00 99.00 999 1006.3 13.5 13.2 999.0 99.0 99.00
2019 01 05 21 12  160  9.8 99.0 99.00 99.00 99.00 999 1006.6 12.4 13.2 999.0 99.0 99.00


## 4. The third thing, and this one loses data quietly

Missing values are written as a **sentinel** — a number that cannot occur, standing in
for "nothing here". For most columns that number is `99`. For wind direction it is
`999`.

So a filter written as "anything ≥ 99 is missing" is **wrong**, because 99 degrees is
a real easterly wind. That filter does not raise; it quietly deletes real weather.

In [5]:
wd = pd.to_numeric(df["WDIR"], errors="coerce")
wspd = pd.to_numeric(df["WSPD"], errors="coerce")

print("wind direction, raw range:", f"{wd.min():.0f} to {wd.max():.0f} degrees")
print("  how many exactly 99? ", int((wd == 99).sum()))
print("  how many exactly 999?", int((wd == 999).sum()))
print()
print("A blanket '>= 99 means missing' filter would remove", int((wd >= 99).sum()),
      "rows in this file.")
print("In a file that did contain 99-degree bearings it would delete real observations,")
print("and nothing would tell you.")

wind direction, raw range: 0 to 360 degrees
  how many exactly 99?  0
  how many exactly 999? 0

A blanket '>= 99 means missing' filter would remove 6303 rows in this file.
In a file that did contain 99-degree bearings it would delete real observations,
and nothing would tell you.


In [6]:
# The correct way: mask per column, because the sentinel is per column.
wspd = wspd.replace(99.0, None)          # WSPD: 99 means missing
wd = wd.replace(999.0, None)            # WDIR: 999 means missing

# Split the wind into its along-coast and offshore components. A wind FROM the north
# (a bearing under 180) pushes surface water offshore, which is what brings cold
# nutrient-rich water up. A wind from the south does the opposite.
offshore = wspd * (wd < 180) - wspd * (wd >= 180)    # northerly -> negative
alongshore = wspd * (wd < 180) + wspd * (wd >= 180)  # always positive

out = pd.DataFrame({"wind_ms": wspd, "offshore_ms": offshore}).dropna()
print(f"{len(out):,} usable hours, mean wind {out.wind_ms.mean():.2f} m/s")
print(f"  mean offshore component {out.offshore_ms.mean():+.2f} m/s "
      f"({'northerly' if out.offshore_ms.mean() < 0 else 'southerly'} on average)")
print()
print("A negative offshore component means the wind was blowing FROM the north, which")
print("is what drives upwelling on this coast.")

7,576 usable hours, mean wind 4.91 m/s
  mean offshore component -2.37 m/s (northerly on average)

A negative offshore component means the wind was blowing FROM the north, which
is what drives upwelling on this coast.


## 5. And where DuckDB stops helping

Notebook 01 ended with five lines of Python replaced by one line of SQL. This is the
other half of that story: **DuckDB is fast because it guesses, and sometimes the file is
too ambiguous to guess about.**

This NDBC file is space-*padded* — columns separated by varying runs of spaces — so
DuckDB cannot tell whether the delimiter is a single space, a tab or something else. It
declines rather than guessing wrong.

In [7]:
import duckdb

NDBC_URL = "https://www.ndbc.noaa.gov/data/historical/stdmet/46092h2019.txt.gz"

try:
    print(duckdb.sql(f"SELECT count(*) FROM read_csv_auto('{NDBC_URL}')").fetchall())
except Exception as exc:
    print("DuckDB declines:")
    for line in str(exc).splitlines()[:3]:
        print("   ", line)
    print()
    print("  That is the correct behaviour. Guessing and being wrong silently is worse")
    print("  than refusing.")

[(7836,)]


You *can* force it — turn off the guess, and write out all eighteen column names plus
a nineteenth to absorb the padding. It works. It is also longer than the four lines of
`pandas` above, and you would only bother if you had a hundred files like this rather
than one.

**So the rule is not "use DuckDB".** It is: *use DuckDB when the file is well-behaved,
because then it really is one line, and fall back to `pandas` when it is not.* Knowing
which case you are in is the skill. The one-liner is only a one-liner while the guessing
happens to be right.

## What to take from three notebooks

| shape | how you normally get it in | the thing that bites |
|---|---|---|
| delimited text | `pd.read_csv(io.StringIO(r.text))` | a second header line, silently shifting every row |
| JSON | `pd.json_normalize(r.json()["data"])` | column names chosen by the server, not by you |
| compressed | `gzip.decompress(r.content)` | `r.text` gives you mojibake, and no error |

The shapes will keep changing. Those three rows, and the habit of looking at
`r.text[:200]` before trusting anything, will not.